# GatorRisk — Demo Notebook
**Clinical NLP Pipeline for Lifestyle Risk Factor Extraction**

Built at UF as an extension of the CTSI NLP Core GatorTron smoking extractor.

This notebook walks through the full pipeline end-to-end on sample notes.
Use this to demo the project to Dr. Wu or include in your portfolio.

---

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import pandas as pd
from modules.pipeline import Pipeline
from modules.preprocessor import ClinicalPreprocessor
from modules.ner_extractor import NERExtractor
from modules.relation_extractor import RelationExtractor
from modules.normalizer import Normalizer
from modules.risk_scorer import RiskScorer

print('GatorRisk modules loaded ✓')

## 1. Run on a Single Note

In [ ]:
note = """
58-year-old male presenting for annual physical.
SOCIAL HISTORY:
Smokes 1.5 packs per day for 30 years.
Drinks 3 beers nightly.
BMI 34.2, class I obesity.
Sedentary lifestyle, no regular exercise.
Sleeps 4-5 hours per night; loud snoring, OSA suspected.
Diet is poor — high sodium, frequent fast food.
Denies illicit drug use.
"""

pipeline = Pipeline(use_transformer=False)
result = pipeline.run_note(note_id='DEMO_001', text=note)

print(result.risk_profile.summary())

## 2. Inspect Each Pipeline Stage

In [ ]:
# Stage 1: Preprocessor
preprocessor = ClinicalPreprocessor(deidentify=True)
processed = preprocessor.process('DEMO_001', note)

print(f'Sentences extracted: {len(processed.sentences)}')
for i, s in enumerate(processed.sentences, 1):
    print(f'  [{i}] {s}')

In [ ]:
# Stage 2: NER
ner = NERExtractor(use_transformer=False)
ner_result = ner.extract('DEMO_001', processed.sentences)

print(f'Entity summary: {ner_result.summary()}')
print()
for e in ner_result.entities:
    print(f'  [{e.label:<20}] {e.sub_label:<30} → "{e.text}"')

In [ ]:
# Stage 3: Relations
rel_extractor = RelationExtractor()
rel_result = rel_extractor.extract('DEMO_001', ner_result)

for r in rel_result.relations:
    print(f'  [{r.factor}] status={r.status} value={r.value} {r.unit or ""} flags={r.flags}')

In [ ]:
# Stage 4: Normalized Profile
normalizer = Normalizer()
profile = normalizer.normalize(rel_result)

print(json.dumps(profile.to_dict(), indent=2))

In [ ]:
# Stage 5: Risk Score
scorer = RiskScorer()
risk = scorer.score(profile)

print(risk.summary())

## 3. Batch Run on 200 Generated Notes

In [ ]:
with open('../data/generated_notes.json') as f:
    notes = json.load(f)

results = pipeline.run_batch(notes)
scores = [r.risk_profile.composite_score for r in results]
tiers  = [r.risk_profile.composite_tier  for r in results]

from collections import Counter
print(f'Notes processed : {len(results)}')
print(f'Avg risk score  : {sum(scores)/len(scores):.3f}')
print(f'Tier breakdown  : {dict(Counter(tiers))}')

In [ ]:
# Simple risk distribution chart (no extra libs needed)
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.hist(scores, bins=20, color='steelblue', edgecolor='white')
plt.xlabel('Composite Risk Score')
plt.ylabel('Count')
plt.title('Risk Score Distribution (200 notes)')

plt.subplot(1, 2, 2)
tier_counts = Counter(tiers)
colors = {'LOW': 'green', 'MODERATE': 'orange', 'HIGH': 'red', 'CRITICAL': 'darkred'}
plt.bar(tier_counts.keys(), tier_counts.values(),
        color=[colors.get(t, 'gray') for t in tier_counts.keys()])
plt.title('Risk Tier Distribution')
plt.ylabel('Count')

plt.tight_layout()
plt.savefig('../data/processed/risk_distribution.png', dpi=150)
plt.show()

## 4. Load MTSamples (once you have the CSV)
Download from: https://www.kaggle.com/datasets/tboyle10/medicaltranscriptions
Place at: `data/mtsamples/mtsamples.csv`

In [ ]:
from pathlib import Path

mt_path = Path('../data/mtsamples/mtsamples.csv')
if mt_path.exists():
    df = pd.read_csv(mt_path)
    print(f'MTSamples loaded: {len(df)} notes')
    print(df['medical_specialty'].value_counts().head(10))

    # Filter to social-history-rich specialties
    relevant = df[df['medical_specialty'].isin([
        'Consult - History and Phy.',
        'General Medicine',
        'SOAP / Chart / Progress Notes',
        'Discharge Summary',
    ])].dropna(subset=['transcription'])

    print(f'\nRelevant notes: {len(relevant)}')

    mt_notes = [
        {'note_id': f'MTS_{i}', 'text': str(row['transcription'])}
        for i, row in relevant.head(50).iterrows()
    ]
    mt_results = pipeline.run_batch(mt_notes)
    mt_scores = [r.risk_profile.composite_score for r in mt_results]
    print(f'\nMTSamples avg risk: {sum(mt_scores)/len(mt_scores):.3f}')
else:
    print('MTSamples CSV not found yet.')
    print('Download from: https://www.kaggle.com/datasets/tboyle10/medicaltranscriptions')

---
## Next Steps
- Get MIMIC-III access at physionet.org for real EHR data
- Enable transformer NER: `Pipeline(use_transformer=True)` on HiPerGator
- Email Dr. Yonghui Wu at yonghui.wu@ufl.edu with this notebook
- Push to GitHub and add to your portfolio